# Module A walkthrough
Fit historical references on approved synthetic lots, then score an unseen lot at 24 hours.

In [ ]:
import pandas as pd
from ess_module_a import ModuleAEngine, default_config
from ess_module_a.synthetic import generate_synthetic_data

In [ ]:
training = generate_synthetic_data(n_lots=12, components_per_lot=60, seed=170)
engine = ModuleAEngine(default_config())
reference = engine.fit(training)
engine.model_info()

In [ ]:
unseen = generate_synthetic_data(n_lots=3, components_per_lot=60, seed=171)
lot = unseen.loc[(unseen.lot_id == 'LOT_001') & (unseen.time_h <= 24)].copy()
lot['lot_id'] = 'DEMO_LOT'
lot['component_id'] = lot.component_id.str.replace('LOT_001', 'DEMO_LOT')
report = engine.score_lot(lot, as_of_h=24)

In [ ]:
components = pd.DataFrame(report['component_results'])
components.status.value_counts()

In [ ]:
parameter_results = pd.DataFrame(report['parameter_results'])
parameter_results.loc[(parameter_results.parameter == 'leakage_current') & (parameter_results.time_h == 24)].sort_values('risk_score', ascending=False).head(10)[['component_id', 'normalized_value', 'static_status', 'robust_z_lot', 'mahalanobis_percentile', 'status', 'reason_codes']]